# 02 — Streaming Ingestion (Auto Loader → Bronze)

Uses Spark Structured Streaming with **Auto Loader** (`cloudFiles`) to
incrementally ingest the JSON files we generated in notebook 01.

Auto Loader automatically detects new files as they land and processes them
exactly once — no custom bookkeeping, no re-processing.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events"
BRONZE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_bronze"
CHECKPOINT_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events/_checkpoints/bronze"

## Stream from JSON files with Auto Loader

This is the **file-based** approach. Auto Loader watches a directory and
picks up new files incrementally — great for cloud storage landing zones.

In [2]:
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .load(VOLUME_PATH)
)

print("Streaming schema:")
raw_stream.printSchema()

Streaming schema:
root
 |-- bet_amount: string (nullable = true)
 |-- casino_floor: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- game_type: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- win_amount: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



## What if your source is Kafka?

If your data comes from **Kafka** instead of files, the change is minimal.
Here's what the equivalent Kafka read looks like:

```python
# ---- Kafka version (swap in for the cloudFiles read above) ----
# raw_stream = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "your-broker:9092")
#     .option("subscribe", "slot_events")
#     .option("startingOffsets", "earliest")
#     .load()
#     .selectExpr("CAST(value AS STRING) as json_str")
#     .select(from_json("json_str", schema).alias("data"))
#     .select("data.*")
# )
```

Everything downstream (Bronze table, Silver transforms, analytics) stays
**exactly the same**. The streaming source is the only thing that changes.

## Write the stream to a Bronze Delta table

In [3]:
query = (
    raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)  # process all available data, then stop
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()
print(f"Stream complete. Data written to {BRONZE_TABLE}")

Stream complete. Data written to alexander_booth.streaming_demo.slot_events_bronze


## Verify Bronze table

In [4]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze table row count: {bronze_df.count()}")
bronze_df.show(10, truncate=False)

Bronze table row count: 500
+----------+------------+------------------------------------+-------------------+--------------------+----------+---------+----------+-------------+
|bet_amount|casino_floor|event_id                            |event_timestamp    |game_type           |machine_id|player_id|win_amount|_rescued_data|
+----------+------------+------------------------------------+-------------------+--------------------+----------+---------+----------+-------------+
|200.34    |High Roller |4e938876-687f-4d0d-ba6b-a89d2506739f|2026-03-25T10:06:35|Electronic Blackjack|MCH-0020  |PLR-23396|0.0       |NULL         |
|353.93    |Sports Zone |7353c767-341a-4c02-a1f4-449696c6de6b|2026-03-25T10:06:30|Electronic Roulette |MCH-0023  |PLR-38785|436.57    |NULL         |
|147.46    |VIP Lounge  |1c2d0580-bc7d-47ee-9706-3d5678232340|2026-03-25T10:06:57|Electronic Roulette |MCH-0026  |PLR-51347|193.88    |NULL         |
|29.75     |VIP Lounge  |6e2d1b3a-61a5-464f-82b9-5d5be128a8b2|2026-03-25